# Arsitektur Broker + Benchmark 4-Channel vs Fused-1-Channel

Notebook pendamping paper JSD-Fuzzy ANN. Menjawab masukan pembimbing: pada
**implementasi**, data 4 sensor sebaiknya dikumpulkan lewat **broker** dulu
(jadi satu, ter-align waktu) baru masuk **classifier (ANN)** — bukan tiap sensor
loncat langsung ke classifier.

Isi:
1. **Diagram arsitektur broker**: 4 sensor -> MQTT broker (agregasi + align) -> dataset gabungan -> ekstraksi fitur -> ANN -> keputusan fault.
2. **Benchmark**: (A) **4-channel** (fitur per-sensor digabung) vs (B) **fused 1-channel** (4 sensor difusikan jadi 1 sinyal rata-rata) — membuktikan mempertahankan 4 channel lebih baik.
3. **Catatan trade-off**.

Model tetap **ANN (MLPClassifier)**. Data, fault-injection, dan fitur (entropy + hybrid time-domain) identik dgn notebook utama supaya sebanding.

In [ ]:
# === Runtime guard ===
import os, time
for _v in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS","VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")
NOTEBOOK_START = time.time()
KAGGLE_TIME_BUDGET_H = float(os.environ.get("KAGGLE_TIME_BUDGET_H", 10.5))
def elapsed_s(): return time.time()-NOTEBOOK_START
def budget_ok(need_s=0.0, label=""):
    left = KAGGLE_TIME_BUDGET_H*3600.0 - elapsed_s()
    if left < need_s:
        print(f"[budget] SKIP {label}: {left/60:.1f} min left"); return False
    return True
def log_stage(x): print(f"[t+{elapsed_s()/60:5.1f} min] {x}", flush=True)
log_stage("guard armed")

In [ ]:
# === Imports + config ===
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
import requests
from io import StringIO
from numpy.lib.stride_tricks import sliding_window_view
from joblib import Parallel, delayed
from scipy import stats as _sstats
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support, roc_auc_score)

RUNTIME_PROFILE = os.environ.get("RUNTIME_PROFILE","fast")
N_JOBS=-1
DS=4; WIN=256; STRIDE=128; MAX_PER_CLASS=200; RANDOM_SEED=42
S=10; scales=np.arange(1,S+1); m=2; r_ratio=0.2; n_ref=128; jsd_bins=40
METHODS=["EDM-Fuzzy","JSD-Fuzzy"]
os.makedirs("exports", exist_ok=True)
print("config: DS",DS,"WIN",WIN,"MAX_PER_CLASS",MAX_PER_CLASS,"S",S,"| profile",RUNTIME_PROFILE)

In [ ]:
# === Load data (4 sensor) — output broker = satu tabel gabungan ===
def load_default_data():
    url="https://raw.githubusercontent.com/vousmeevoyez/public-files/refs/heads/main/tabel_sensor4_generated.csv"
    r=requests.get(url); r.raise_for_status()
    return pd.read_csv(StringIO(r.text))
df=load_default_data()
cols=["kelembaban1","kelembaban2","kelembaban3","kelembaban4"]
X_df=pd.DataFrame(df[cols].to_numpy(dtype=float),columns=cols).ffill().bfill()
X_df=X_df.fillna(X_df.median(numeric_only=True))
X=X_df.to_numpy(); X_ds=X[::DS]
print("Combined table (broker output):",X.shape,"-> downsampled",X_ds.shape)

In [ ]:
# === Fault simulators + skenario (identik notebook utama) ===
def simulate_drift_fault(x,intensity=0.02,seed=None):
    drift=np.arange(len(x))*intensity; return x+drift, np.abs(drift)>1e-6
def simulate_spike_fault(x,intensity=0.08,p=0.015,seed=None):
    tau=max(1,int(1.0/p)) if p>0 else len(x)
    spikes=(np.arange(len(x))%tau==0).astype(float)*(intensity*np.nanstd(x))
    return x+spikes, spikes!=0
def simulate_bias_fault(x,bias=0.08,seed=None):
    return x+bias, np.ones(len(x),bool)
def simulate_hardware_fault(x,stuck_prob=0.08,loss_prob=0.05,seed=None):
    rng=np.random.default_rng(seed); n=len(x); rv=rng.random(n); idx=rng.integers(n,size=n)
    m1=rv<stuck_prob; y=x.copy(); y[m1]=x[idx[m1]]; m2=rv<loss_prob; y[m2]=np.nan
    return y,(m1|m2)
def simulate_multiple_faults(x,faults,seed=None):
    y=x.copy(); m=np.zeros(len(x),bool)
    for f,kw in faults:
        y,mi=f(y,**kw,seed=seed); m|=mi
    return y,m
def simulate_choose_one(x,options,seed=None):
    rng=np.random.default_rng(seed); f,kw=options[rng.integers(len(options))]
    return f(x,**kw,seed=seed)
SCENARIOS={
 "faulty":[(simulate_choose_one,{"options":[(simulate_drift_fault,{"intensity":0.02}),(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_bias_fault,{"bias":0.08}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})]})],
 "drift":[(simulate_drift_fault,{"intensity":0.02})],
 "spike":[(simulate_spike_fault,{"intensity":0.08,"p":0.015})],
 "bias":[(simulate_bias_fault,{"bias":0.08})],
 "hardware":[(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "bias+malfunc":[(simulate_bias_fault,{"bias":0.08}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "spike+malfunc":[(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "spike+bias":[(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_bias_fault,{"bias":0.08})],
 "drift+malfunc":[(simulate_drift_fault,{"intensity":0.02}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "drift+bias":[(simulate_drift_fault,{"intensity":0.02}),(simulate_bias_fault,{"bias":0.08})],
 "drift+spike":[(simulate_drift_fault,{"intensity":0.02}),(simulate_spike_fault,{"intensity":0.08,"p":0.015})],
 "spike+bias+malfunc":[(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_bias_fault,{"bias":0.08}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "drift+bias+malfunc":[(simulate_drift_fault,{"intensity":0.02}),(simulate_bias_fault,{"bias":0.08}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "spike+drift+malfunc":[(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_drift_fault,{"intensity":0.02}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "drift+spike+bias":[(simulate_drift_fault,{"intensity":0.02}),(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_bias_fault,{"bias":0.08})],
 "spike+bias+malfunc+drift":[(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_bias_fault,{"bias":0.08}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05}),(simulate_drift_fault,{"intensity":0.02})],
}
print("skenario:",len(SCENARIOS))

In [ ]:
# === Windowing + build dataset (N, WIN, 4) ===
def make_windows(Xn,win,stride):
    Xn=np.asarray(Xn,dtype=np.float32); N=Xn.shape[0]
    if N<win: return np.empty((0,win,Xn.shape[1]),np.float32), np.array([],int)
    view=sliding_window_view(Xn,window_shape=win,axis=0); starts=np.arange(0,N-win+1,stride,dtype=int)
    return view[starts], starts
def inject_faults_multisensor(Xin,scenario_faults,seed=0):
    rng=np.random.default_rng(seed); Y=Xin.copy(); M=np.zeros_like(Y,bool)
    for s in range(Y.shape[1]):
        y,mm=simulate_multiple_faults(Y[:,s],scenario_faults,seed=int(rng.integers(1e9))); Y[:,s]=y; M[:,s]=mm
    Ydf=pd.DataFrame(Y).ffill().bfill(); Ydf=Ydf.fillna(Ydf.median(numeric_only=True))
    return Ydf.to_numpy(), M
def window_fault_label(mask,win,stride,thr=0.02):
    T=len(mask)
    if win>T: return np.zeros(0,bool), np.array([],int)
    Wm=sliding_window_view(mask,window_shape=win,axis=0)[::stride]; ratio=Wm.mean(axis=1)
    return (ratio>thr).any(axis=1), np.arange(0,T-win+1,stride,dtype=int)

datasets=[]; labels=[]; scenario_names=["normal"]+list(SCENARIOS.keys())
W0,_=make_windows(X_ds,WIN,STRIDE); datasets.append(W0); labels.append(np.zeros(len(W0),int))
for k,(name,faults) in enumerate(SCENARIOS.items(),start=1):
    Y,Mk=inject_faults_multisensor(X_ds,faults,seed=100+k)
    isf,_=window_fault_label(Mk,WIN,STRIDE); Wk,_=make_windows(Y,WIN,STRIDE); Wk=Wk[isf]
    datasets.append(Wk); labels.append(np.full(len(Wk),k,int))
W_all=np.concatenate(datasets,axis=0); y_all=np.concatenate(labels,axis=0)
if W_all.ndim==3 and W_all.shape[1]==4 and W_all.shape[2]==WIN: W_all=W_all.transpose(0,2,1)
def balanced_subsample(Xw,y,maxp,seed=0):
    rng=np.random.default_rng(seed); keep=[]
    for c in np.unique(y):
        idx=np.where(y==c)[0]
        if len(idx)>maxp: idx=rng.choice(idx,size=maxp,replace=False)
        keep.append(idx)
    keep=np.concatenate(keep); rng.shuffle(keep); return Xw[keep],y[keep]
W_s,y_s=balanced_subsample(W_all,y_all,MAX_PER_CLASS,RANDOM_SEED)
print("W_s",W_s.shape,"| classes",len(np.unique(y_s)))

In [ ]:
# === Entropy (EDM-Fuzzy + JSD-Fuzzy) — identik notebook utama ===
def coarse_grain_mean(x,s):
    n=(len(x)//s)*s
    return x[:n].reshape(-1,s).mean(axis=1) if n>0 else np.array([],float)
def embed_matrix(y,mm):
    L=len(y)
    return np.lib.stride_tricks.sliding_window_view(y,mm) if L>=mm else np.empty((0,mm),float)
def fuzzy_phi(V,r,n_ref=256,seed=0):
    rng=np.random.default_rng(seed); N=V.shape[0]
    if N<3: return np.nan
    ref=rng.choice(N,size=n_ref,replace=False) if N>n_ref else np.arange(N)
    A=V[ref]; a2=np.sum(A*A,1,keepdims=True); b2=np.sum(V*V,1,keepdims=True).T
    d2=np.maximum(a2+b2-2*(A@V.T),0.0); mu=1.0/(1.0+d2/(r*r+1e-24)); mu[np.arange(len(ref)),ref]=0.0
    return (mu.sum(1)/(N-1)).mean()
def fuzzy_similarity_samples(V,r,n_ref=256,seed=0):
    rng=np.random.default_rng(seed); N=V.shape[0]
    if N<3: return np.array([],float)
    ref=rng.choice(np.arange(N),size=n_ref,replace=False) if N>n_ref else np.arange(N)
    A=V[ref]; a2=np.sum(A*A,1,keepdims=True); b2=np.sum(V*V,1,keepdims=True).T
    d=np.sqrt(np.maximum(a2+b2-2*(A@V.T),0.0)); mu=1.0/(1.0+(d/(r+1e-12))**2)
    for ri,i in enumerate(ref): mu[ri,i]=np.nan
    return mu[~np.isnan(mu)].ravel()
def edm_fuzzy_entropy_1d(x,scales,m=2,r_ratio=0.2,n_ref=256,seed=0):
    out=[]
    for s in scales:
        y=coarse_grain_mean(x,s)
        if len(y)<(m+2): out.append(np.nan); continue
        r=r_ratio*np.std(y,ddof=1)
        pm=fuzzy_phi(embed_matrix(y,m),r,n_ref,seed+11*s); pm1=fuzzy_phi(embed_matrix(y,m+1),r,n_ref,seed+17*s)
        out.append(np.log(pm/pm1) if (pm and pm1 and pm>0 and pm1>0 and not np.isnan(pm) and not np.isnan(pm1)) else np.nan)
    return np.array(out,float)
def jsd_fuzzy_entropy_1d(x,scales,m=2,r_ratio=0.2,n_ref=256,seed=0,bins=20,rich=True):
    out=[]; per=4 if rich else 1; be=np.linspace(0,1,bins+1); eps=1e-12
    for s in scales:
        y=coarse_grain_mean(x,s)
        if len(y)<(m+2): out.extend([np.nan]*per); continue
        r=r_ratio*np.std(y,ddof=1)
        mu_m=fuzzy_similarity_samples(embed_matrix(y,m),r,n_ref,seed+11*s)
        mu_m1=fuzzy_similarity_samples(embed_matrix(y,m+1),r,n_ref,seed+17*s)
        if len(mu_m)==0 or len(mu_m1)==0: out.extend([np.nan]*per); continue
        p,_=np.histogram(mu_m,bins=be); q,_=np.histogram(mu_m1,bins=be); p=p.astype(float); q=q.astype(float)
        if p.sum()==0 or q.sum()==0: out.extend([np.nan]*per); continue
        p/=p.sum(); q/=q.sum(); mid=0.5*(p+q)
        jsd=0.5*(np.sum(p*np.log((p+eps)/(mid+eps)))+np.sum(q*np.log((q+eps)/(mid+eps))))
        if rich: out.extend([jsd,np.log((mu_m.mean()+eps)/(mu_m1.mean()+eps)),mu_m.mean(),mu_m.std()])
        else: out.append(jsd)
    return np.array(out,float)
def compute_features_entropy(W,scales,method,m=2,r_ratio=0.2,n_ref=256,jsd_bins=20,seed=0,n_jobs=-1):
    Nwin,win,ns=W.shape; mk=method.strip().lower()
    def ent(x,sd):
        if mk=='edm-fuzzy': return edm_fuzzy_entropy_1d(x,scales,m,r_ratio,n_ref,sd)
        if mk=='jsd-fuzzy': return jsd_fuzzy_entropy_1d(x,scales,m,r_ratio,n_ref,sd,jsd_bins)
        raise ValueError(method)
    def one(i): return np.concatenate([ent(W[i,:,s],seed+1000*i+19*s) for s in range(ns)])
    if n_jobs==1 or Nwin<=1: F=np.vstack([one(i) for i in range(Nwin)])
    else: F=np.vstack(Parallel(n_jobs=n_jobs,prefer='processes')(delayed(one)(i) for i in range(Nwin)))
    Fdf=pd.DataFrame(F); Fdf=Fdf.fillna(Fdf.median(numeric_only=True)).fillna(0.0)
    return Fdf.to_numpy()
print("entropy fns ready")

In [ ]:
# === Time-domain (hybrid) features — any channel count ===
def compute_time_features(W):
    N,WINl,ns=W.shape; t=np.arange(WINl, dtype=float); tc=t-t.mean(); tv=(tc**2).mean()+1e-12; feats=[]
    for s in range(ns):
        x=np.asarray(W[:,:,s],float); mu=x.mean(1); sd=x.std(1); rms=np.sqrt((x**2).mean(1))
        ptp=x.max(1)-x.min(1); mad=np.abs(x-mu[:,None]).mean(1)
        sk=_sstats.skew(x,axis=1,bias=False); ku=_sstats.kurtosis(x,axis=1,bias=False)
        d=np.diff(x,axis=1); sm=np.abs(d).mean(1); smx=np.abs(d).max(1)
        tr=(x*tc[None,:]).mean(1)/tv; sc=np.sign(x-mu[:,None]); zcr=(np.abs(np.diff(sc,axis=1))>0).mean(1)
        xf=np.abs(np.fft.rfft(x-mu[:,None],axis=1)); pw=xf**2; half=max(1,pw.shape[1]//2)
        hf=pw[:,half:].sum(1)/(pw.sum(1)+1e-12)
        feats.extend([mu,sd,rms,ptp,mad,sk,ku,sm,smx,tr,zcr,hf])
    return np.nan_to_num(np.column_stack(feats),nan=0.0,posinf=0.0,neginf=0.0)
print("time fns ready")

In [ ]:
# === Bangun fitur: (A) 4-channel vs (B) fused 1-channel ===
# A: pertahankan 4 sensor -> fitur per-sensor digabung (seperti notebook utama).
# B: fusikan 4 sensor jadi 1 sinyal (rata-rata per waktu) -> fitur 1 channel.
W_A = W_s                                  # (N, WIN, 4)
W_B = W_s.mean(axis=2, keepdims=True)       # (N, WIN, 1) — fused single stream
print("A (4-channel):", W_A.shape, "| B (fused 1-channel):", W_B.shape)

def build_hybrid(W, method):
    Fe = compute_features_entropy(W, scales, method, m, r_ratio, n_ref, jsd_bins, seed=7, n_jobs=N_JOBS)
    T  = compute_time_features(W)
    return np.hstack([Fe, T])

VARIANTS = {"A_4channel": W_A, "B_fused_1ch": W_B}
FEAT = {}
for meth in METHODS:
    for vname, Wv in VARIANTS.items():
        log_stage(f"features {meth} / {vname}")
        FEAT[(meth, vname)] = build_hybrid(Wv, meth)
        print(f"  {meth:10s} {vname:12s} -> {FEAT[(meth,vname)].shape}")

In [ ]:
# === Target multi-label per-tipe fault ===
BASE_FAULTS=["drift","spike","bias","hardware"]
def scenario_to_multilabel(name):
    toks=name.replace("malfunc","hardware").split("+")
    return [1 if b in toks else 0 for b in BASE_FAULTS]
_excl=scenario_names.index("faulty") if "faulty" in scenario_names else -1
keep_ml=np.where(y_s!=_excl)[0]
Y_multi=np.array([scenario_to_multilabel(scenario_names[int(c)]) for c in y_s[keep_ml]])
print("multi-label windows:",len(keep_ml),"| prevalensi:",{b:round(float(Y_multi[:,j].mean()),2) for j,b in enumerate(BASE_FAULTS)})

In [ ]:
# === Benchmark: deteksi per-fault, A(4ch) vs B(fused) ===
def detect(X, yb, seed=42):
    Xtr,Xte,ytr,yte=train_test_split(X,yb,test_size=0.25,random_state=seed,stratify=yb)
    I=X.shape[1]
    pipe=Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),
                   ("mlp",MLPClassifier(max_iter=300,random_state=seed,early_stopping=True,n_iter_no_change=12))])
    grid={"mlp__hidden_layer_sizes":[(max(16,I//2),),(I,),(max(16,I//2),max(8,I//4))],"mlp__alpha":[1e-4,1e-3]}
    gs=GridSearchCV(pipe,grid,cv=3,scoring="f1",n_jobs=N_JOBS); gs.fit(Xtr,ytr)
    b=gs.best_estimator_; pred=b.predict(Xte); proba=b.predict_proba(Xte)[:,1]
    p,r,f,_=precision_recall_fscore_support(yte,pred,average="binary",zero_division=0)
    try: auc=roc_auc_score(yte,proba)
    except Exception: auc=np.nan
    return dict(Accuracy=accuracy_score(yte,pred),Precision=p,Recall=r,F1=f,ROC_AUC=auc)

rows=[]
for meth in METHODS:
    for vname in VARIANTS:
        Xfull=FEAT[(meth,vname)][keep_ml]
        for j,b in enumerate(BASE_FAULTS):
            yb=Y_multi[:,j]
            if yb.sum()<10 or (len(yb)-yb.sum())<10: continue
            if not budget_ok(120,f"{meth}/{vname}/{b}"): continue
            r=detect(Xfull,yb)
            rows.append({"Method":meth,"Variant":vname,"Fault":b,"n_features":Xfull.shape[1],
                         **{k:round(float(v),3) for k,v in r.items()}})
    log_stage(f"benchmark {meth} done")
bench=pd.DataFrame(rows)
print("\n=== Benchmark per-fault: 4-channel (A) vs fused-1ch (B) ===")
print(bench.to_string(index=False))
bench.to_csv("exports/broker_channel_benchmark.csv",index=False)
print("\n[Saved] exports/broker_channel_benchmark.csv")
# ringkasan rata-rata per (Method,Variant)
summ=bench.groupby(["Method","Variant"])[["Accuracy","F1","ROC_AUC"]].mean().round(3)
print("\n=== Rata-rata per metode x variant (4 fault) ==="); print(summ.to_string())
summ.to_csv("exports/broker_channel_summary.csv")
display(bench)

In [ ]:
# === Plot: F1 per-fault, 4-channel vs fused (per metode) ===
fig,axes=plt.subplots(1,len(METHODS),figsize=(13,5),sharey=True)
if len(METHODS)==1: axes=[axes]
for ax,meth in zip(axes,METHODS):
    sub=bench[bench.Method==meth]
    piv=sub.pivot(index="Fault",columns="Variant",values="F1").reindex(BASE_FAULTS)
    piv.columns=[("A: 4-channel" if "4channel" in c else "B: fused 1-ch") for c in piv.columns]
    piv.plot(kind="bar",ax=ax,ylim=(0,1.05),width=0.75)
    ax.set_title(f"{meth}: deteksi per-fault"); ax.set_ylabel("F1"); ax.set_xlabel("Tipe fault")
    ax.grid(True,axis="y",alpha=0.3); ax.tick_params(axis="x",rotation=0)
    for c in ax.containers: ax.bar_label(c,fmt="%.2f",fontsize=8)
plt.suptitle("Benchmark Arsitektur: 4-Channel (pertahankan sensor) vs Fused 1-Channel",fontweight="bold")
plt.tight_layout(); plt.savefig("exports/broker_channel_benchmark.png",dpi=150,bbox_inches="tight"); plt.show()
print("[Saved] exports/broker_channel_benchmark.png")

In [ ]:
# === Diagram Arsitektur Broker ===
from matplotlib.patches import FancyBboxPatch
fig,ax=plt.subplots(figsize=(13,6)); ax.axis("off"); ax.set_xlim(0,14); ax.set_ylim(0,10)
def box(x,y,w,h,txt,fc,fs=10):
    ax.add_patch(FancyBboxPatch((x-w/2,y-h/2),w,h,boxstyle="round,pad=0.03,rounding_size=0.12",
        facecolor=fc,edgecolor="black",lw=1.3))
    ax.text(x,y,txt,ha="center",va="center",fontsize=fs,fontweight="bold",color="white")
def arrow(x0,y0,x1,y1):
    ax.annotate("",xy=(x1,y1),xytext=(x0,y0),arrowprops=dict(arrowstyle="-|>",lw=2,color="#444"))
SC="#3b6fb0"; BR="#b0562b"; DS="#7a4fb0"; FE="#e8892b"; AN="#4e9a51"; OUT="#c0392b"
sy=[8.6,6.9,5.2,3.5]
for i,y in enumerate(sy):
    box(1.6,y,2.2,1.05,f"Sensor {i+1}\nkelembaban{i+1}",SC,9)
    arrow(2.75,y,4.35,6.05)
box(5.4,6.05,2.1,2.4,"MQTT\nBROKER\n(agregasi +\nalign waktu)",BR,10)
arrow(6.5,6.05,7.55,6.05)
box(8.7,6.05,2.3,1.6,"Dataset\nGABUNGAN\n(1 tabel, 4 kolom)",DS,9)
arrow(9.9,6.05,10.9,6.05)
box(11.9,7.3,2.0,1.3,"Ekstraksi Fitur\n(entropy +\ntime-domain)",FE,9)
arrow(11.9,6.6,11.9,5.75)
box(11.9,4.7,2.0,1.3,"ANN\n(MLPClassifier)",AN,10)
arrow(11.9,4.0,11.9,3.15)
box(11.9,2.4,2.0,1.1,"Keputusan Fault\n(per-tipe)",OUT,9)
ax.text(7,9.5,"Arsitektur Implementasi: 4 Sensor -> Broker -> Classifier (ANN)",
        ha="center",fontsize=14,fontweight="bold")
ax.text(5.4,4.55,"'jangan loncat': kumpulkan\ndulu di broker, baru classify",
        ha="center",fontsize=8.5,style="italic",color="#555")
plt.tight_layout(); plt.savefig("exports/broker_architecture_diagram.png",dpi=150,bbox_inches="tight"); plt.show()
print("[Saved] exports/broker_architecture_diagram.png")

## Catatan Trade-off — kenapa pertahankan 4 channel

**Peran broker.** Broker (mis. MQTT) = lapisan pengumpul: 4 stream sensor
di-*subscribe*, di-align waktu, disatukan jadi **satu dataset gabungan**
(`tabel_sensor4_generated.csv` = output broker), lalu baru masuk pipeline
fitur + ANN. Ini menjawab "jangan loncat": sensor tidak langsung ke classifier.

**Menyatukan data (broker) != menyatukan sinyal.** Dua makna "digabung jadi satu":

| | (A) 4-channel — *dipakai* | (B) fused 1-channel |
|---|---|---|
| Broker | kumpulkan 4 stream jadi 1 dataset (4 kolom) | kumpulkan lalu **rata-rata jadi 1 sinyal** |
| Info per-sensor | **dipertahankan** | **hilang** |
| Fault lokal (spike/hardware di 1 sensor & waktu acak) | terdeteksi | **ter-encer** oleh rata-rata |
| Dimensi fitur | 4x lebih besar | kecil (lebih murah) |
| Akurasi deteksi | **lebih tinggi** (lihat benchmark) | lebih rendah utk fault lokal |

**Kesimpulan.** Broker tetap dipakai untuk *mengumpulkan* data jadi satu
(sesuai masukan pembimbing), **tetapi 4 channel dipertahankan** masuk ke ANN —
bukan difusikan jadi 1 sinyal. Benchmark di atas menunjukkan fusi 1-channel
menurunkan F1 deteksi (terutama spike & hardware yang timing-nya per-sensor),
sehingga arsitektur `4 sensor -> broker (agregasi) -> dataset 4-kolom -> ANN`
adalah pilihan yang benar.

Model **tetap ANN**. Yang ditambah hanya lapisan **broker/agregasi** eksplisit
pada narasi implementasi — pipeline ML tidak berubah.